# Rodada 2 — auditoria antes dos modelos

In [1]:
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d
print(len(d),"pares na grade; ver ESTRATEGIA.md para escolhas anteriores à estimação.")

3384 pares na grade; ver ESTRATEGIA.md para escolhas anteriores à estimação.


## 1. Unidades, temporalidade e seleção

Preços já são lag1. Não há preço corrente ou fonte mensal para reconfirmar a defasagem original. GAEZ terras2010 não é estritamente predeterminado. Taxas são frações; VD linear usa pp. Contagem parcial e faltante não é zero.

In [2]:
assert len(d)==3384 and d.id.nunique()==141
assert d.ano.min()==2001 and d.ano.max()==2024
assert not d.duplicated(['id','ano']).any()
cols=['pp_agricultura','pp_pastagem','estoque','p_soja','p_milho','p_boi','a_soja','a_milho','autos','fiscal']
r=d[cols].describe(percentiles=[.01,.25,.5,.75,.99]).T
r['faltantes']=d[cols].isna().sum();r['zeros']=d[cols].eq(0).sum();r.to_csv(e.p/'auditoria_distribuicoes.csv')
suporte=d.groupby('bioma').agg(municipios=('id','nunique'),pares=('ano','size'),conversao=('pp_agricultura','count'),fiscal=('fiscal','count'))
display(suporte);suporte.to_csv(e.p/'auditoria_suporte.csv')
s=d.dropna(subset=['pp_agricultura','pp_pastagem','expo_soja','expo_milho'])
selecionado=s.fiscal.notna()
comparacao=s.groupby(selecionado)[['pp_agricultura','pp_pastagem','estoque','z_a_soja','z_a_milho']].agg(['count','mean','median'])
comparacao.to_csv(e.p/'auditoria_selecao.csv');display(comparacao)
d.groupby('ano')[cols].count().to_csv(e.p/'auditoria_cobertura_ano.csv')

,municipios,pares,conversao,fiscal
bioma,,,,
Amazônia,76,1824,1306,873
Cerrado,61,1464,1412,519
Pantanal,4,96,54,52


pp_agricultura                     pp_pastagem                      \
                count      mean    median       count      mean    median   
fiscal                                                                      
False            1632  0.137289  0.013751        1632  1.351057  0.914987   
True             1140  0.076364  0.015293        1140  0.963122  0.671353   

       estoque                               z_a_soja                      \
         count           mean         median    count      mean    median   
fiscal                                                                      
False     1632  270179.027967  143711.467084     1632  0.242040  0.186080   
True      1140  493661.066390  294660.742456     1140 -0.214082 -0.368172   

       z_a_milho                      
           count      mean    median  
fiscal                                
False       1632 -0.046033 -0.194274  
True        1140 -0.115529 -0.313659

## 2. Absorção e denominação das exposições

Não há capacidade pecuária. Razões de preço são dimensionais após converter carne para USD/t. A variação das exposições após FE precisa existir.

In [3]:
cols=['z_p_soja','z_p_milho','z_p_boi','z_a_soja','z_a_milho','expo_soja','expo_milho','x_rel_soja','x_rel_milho']
s=d.dropna(subset=['pp_agricultura',*cols])
v,rank,it=projetar(s,cols)
a=pd.DataFrame({'variavel':cols,'sd_apos_FE':v.std(axis=0)});a.to_csv(e.p/'auditoria_absorcao.csv',index=False);display(a)
assert np.max(abs(v[:,:5]))<1e-8
assert np.linalg.matrix_rank(v[:,5:7])==2
corr=s.drop_duplicates('ano')[['p_soja','p_milho','p_boi']].corr();display(corr);corr.to_csv(e.p/'auditoria_correlacao_precos.csv')
print('Estoque evolui endogenamente; não usar offset como instrumento nem prova de exogeneidade.')

,variavel,sd_apos_FE
0,z_p_soja,3.287553e-14
1,z_p_milho,3.010085e-14
2,z_p_boi,3.866350e-14
3,z_a_soja,4.691040e-31
4,z_a_milho,5.512824e-30
5,expo_soja,1.023738e+00
6,expo_milho,8.535998e-01
7,x_rel_soja,1.015818e+00
8,x_rel_milho,8.620818e-01


,p_soja,p_milho,p_boi
p_soja,1.000000,0.947714,0.563342
p_milho,0.947714,1.000000,0.537493
p_boi,0.563342,0.537493,1.000000


Estoque evolui endogenamente; não usar offset como instrumento nem prova de exogeneidade.
